In [1]:
# 1. 데이터 생성

In [2]:
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document


load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
llm.invoke('안녕하세요')

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 9, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVFd2PkeWS2etRR5tLCtNbuDIoTRj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9626-c2cd-7e71-bdd7-4380ed6c23cc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 10, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
import FinanceDataReader as fdr
etf_listing =fdr.StockListing('ETF/KR')

In [5]:
!pwd

/data/lecture/codeit/quantization/soma_lectures


In [6]:
etf_listing.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,94710,2,2045,2.21,94689.0,32.4893,15401829,1449982,218496
1,360750,4,TIGER 미국S&P500,25780,2,265,1.04,25695.0,0.8957,17095776,439893,159784
2,396500,2,TIGER 반도체TOP10,35935,2,445,1.25,36079.0,48.8674,13968697,498769,96809
3,133690,4,TIGER 미국나스닥100,171995,2,3245,1.92,171121.0,1.4622,686569,117768,87029
4,379800,4,KODEX 미국S&P500,23560,2,245,1.05,23485.0,0.9205,22846526,537199,84404


In [7]:
etf_listing['Symbol'].astype(str).str.zfill(6)

0       069500
1       360750
2       396500
3       133690
4       379800
         ...  
1088    334700
1089    253160
1090    465620
1091    306520
1092    301410
Name: Symbol, Length: 1093, dtype: object

In [8]:
name_by_ticker = dict(zip(etf_listing['Symbol'].astype(str).str.zfill(6), etf_listing['Name']))
name_by_ticker

{'069500': 'KODEX 200',
 '360750': 'TIGER 미국S&P500',
 '396500': 'TIGER 반도체TOP10',
 '133690': 'TIGER 미국나스닥100',
 '379800': 'KODEX 미국S&P500',
 '102110': 'TIGER 200',
 '459580': 'KODEX CD금리액티브(합성)',
 '488770': 'KODEX 머니마켓액티브',
 '122630': 'KODEX 레버리지',
 '229200': 'KODEX 코스닥150',
 '278530': 'KODEX 200TR',
 '379810': 'KODEX 미국나스닥100',
 '411060': 'ACE KRX금현물',
 '310970': 'TIGER MSCI Korea TR',
 '357870': 'TIGER CD금리투자KIS(합성)',
 '091160': 'KODEX 반도체',
 '233740': 'KODEX 코스닥150레버리지',
 '498400': 'KODEX 200타겟위클리커버드콜',
 '423160': 'KODEX KOFR금리액티브(합성)',
 '381180': 'TIGER 미국필라델피아반도체나스닥',
 '381170': 'TIGER 미국테크TOP10 INDXX',
 '273130': 'KODEX 종합채권(AA-이상)액티브',
 '148020': 'RISE 200',
 '360200': 'ACE 미국S&P500',
 '458730': 'TIGER 미국배당다우존스',
 '0043B0': 'TIGER 머니마켓액티브',
 '367380': 'ACE 미국나스닥100',
 '481050': 'KODEX CD1년금리플러스액티브(합성)',
 '102780': 'KODEX 삼성그룹',
 '161510': 'PLUS 고배당주',
 '455890': 'RISE 머니마켓액티브',
 '292150': 'TIGER 코리아TOP10',
 '449170': 'TIGER KOFR금리액티브(합성)',
 '395160': 'KODEX AI반도체',
 '487240': 'K

In [53]:
tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
    "132030": {"category": "원자재",   "expense_ratio": 0.68, "dividend_yield": 0.0,
               "keywords": ["금", "원자재", "인플레이션", "헤지", "안전자산"]},
    "167860": {"category": "채권",     "expense_ratio": 0.15, "dividend_yield": 2.8,
               "keywords": ["채권", "국고채", "안전", "이자", "저위험"]},
    "214980": {"category": "채권",     "expense_ratio": 0.05, "dividend_yield": 3.2,
               "keywords": ["단기채", "안전", "예금대안", "저위험", "이자"]},
    "329200": {"category": "부동산",   "expense_ratio": 0.29, "dividend_yield": 4.2,
               "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"]},
    "284430": {"category": "혼합",     "expense_ratio": 0.09, "dividend_yield": 1.5,
               "keywords": ["혼합", "자산배분", "균형", "중위험", "분산투자"]},
    "371460": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["중국", "전기차", "테마", "해외주식", "고위험"]},
}

In [54]:
for t, info in tickers_info.items():
    name = name_by_ticker.get(t, "(이름 미확인)")
    print(f" {t} | {name:30s} | {info['category']}")

 069500 | KODEX 200                      | 국내주식
 379800 | KODEX 미국S&P500                 | 해외주식
 411060 | ACE KRX금현물                     | 배당
 305540 | TIGER 2차전지테마                   | 테마
 379810 | KODEX 미국나스닥100                 | 해외주식
 381180 | TIGER 미국필라델피아반도체나스닥            | 해외주식
 132030 | KODEX 골드선물(H)                  | 원자재
 167860 | KIWOOM 국고채10년레버리지              | 채권
 214980 | KODEX 단기채권PLUS                 | 채권
 329200 | TIGER 리츠부동산인프라                 | 부동산
 284430 | KODEX 200미국채혼합                 | 혼합
 371460 | TIGER 차이나전기차SOLACTIVE          | 해외주식


In [55]:
def collect_etf_price(ticker, days= 365):
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = ( datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    df = fdr.DataReader(ticker, start_date, end_date)
    return {'status' : 'real', 'data' : df, 'ticker' : ticker}

In [56]:
def compute_metrics(df):
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 -1)*100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return {'return_1y' : round(ann_ret, 2), 'volatility' : round(ann_vol,2), 'mdd' : round(mdd, 2)}

In [57]:
def risk_from_vol(vol):
    if vol < 5 : return "낮음"
    if vol < 15: return "약간 낮음"
    if vol < 25: return "중간"
    return "높음"

In [58]:
tickers_info

{'069500': {'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식']},
 '379800': {'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식']},
 '411060': {'category': '배당',
  'expense_ratio': 0.01,
  'dividend_yield': 3.5,
  'keywords': ['미국', '배당', '배당성장', '저비용', '안정']},
 '305540': {'category': '테마',
  'expense_ratio': 0.45,
  'dividend_yield': 0.0,
  'keywords': ['2차전지', '배터리', '테마', '성장', '고위험']},
 '379810': {'category': '해외주식',
  'expense_ratio': 0.07,
  'dividend_yield': 0.0,
  'keywords': ['미국', '나스닥', '기술주', '성장', 'IT']},
 '381180': {'category': '해외주식',
  'expense_ratio': 0.49,
  'dividend_yield': 0.0,
  'keywords': ['반도체', 'AI', '미국', '기술주', '고위험']},
 '132030': {'category': '원자재',
  'expense_ratio': 0.68,
  'dividend_yield': 0.0,
  'keywords': ['금', '원자재', '인플레이션', '헤지', '안전자산']},
 '167860': {'category': '채권',
  'expense_ratio': 0.15,
  'dividend_yiel

In [59]:
etf_data =[]
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    # {'status' : 'real', 'data' : df, 'ticker' : ticker}
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker" : t,
        "name" : name,
        **info,
        **metrics,
        'risk_level' : risk_from_vol(metrics['volatility']),
        'data_staus' : result['status']
    })

In [60]:
etf_data[:2]

[{'ticker': '069500',
  'name': 'KODEX 200',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
  'return_1y': np.float64(227.44),
  'volatility': np.float64(36.58),
  'mdd': np.float64(-20.62),
  'risk_level': '높음',
  'data_staus': 'real'},
 {'ticker': '379800',
  'name': 'KODEX 미국S&P500',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식'],
  'return_1y': np.float64(40.68),
  'volatility': np.float64(13.33),
  'mdd': np.float64(-5.7),
  'risk_level': '약간 낮음',
  'data_staus': 'real'}]

In [61]:
def generate_description(etf):
    
    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}$
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%
    """
    
    return llm.invoke([{'role' : 'user', 'content' : prompt}]).content.strip()

In [62]:
documents = []
for etf in etf_data:
    desc = generate_description(etf)
    text = (f"{etf['name']} ({etf['category']}): {desc} "
            f"키워드: {', '.join(etf['keywords'])}"
            f"수수료 {etf['expense_ratio']}%, 배당수익률: {etf['dividend_yield']}%"
            f"수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%")
    
    metadata = {k:v for k, v in etf.items() if k!="keywords"}
    metadata['keywords'] = ', '.join(etf['keywords'])
    
    documents.append(Document(page_content=text, metadata = metadata))
    

In [84]:
documents

[Document(metadata={'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense_ratio': 0.15, 'dividend_yield': 1.8, 'return_1y': np.float64(227.44), 'volatility': np.float64(36.58), 'mdd': np.float64(-20.62), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '코스피, 대형주, 인덱스, 분산투자, 국내주식'}, page_content='KODEX 200 (국내주식): KODEX 200은 코스피 대형주에 투자하는 국내주식 인덱스 ETF로, 분산투자를 통해 안정성을 추구하며, 1년 수익률이 227.44%로 높은 성과를 기록했습니다. 수수료는 0.15%이며, 배당수익률은 1.8%입니다. 키워드: 코스피, 대형주, 인덱스, 분산투자, 국내주식수수료 0.15%, 배당수익률: 1.8%수익률 : 227.44% / 변동성: 36.58% / MDDD : -20.62%'),
 Document(metadata={'ticker': '379800', 'name': 'KODEX 미국S&P500', 'category': '해외주식', 'expense_ratio': 0.05, 'dividend_yield': 0.0, 'return_1y': np.float64(40.68), 'volatility': np.float64(13.33), 'mdd': np.float64(-5.7), 'risk_level': '약간 낮음', 'data_staus': 'real', 'keywords': '미국, S&P500, 대형주, 성장, 해외주식'}, page_content='KODEX 미국S&P500 (해외주식): KODEX 미국S&P500 ETF는 미국의 S&P500 지수를 추종하는 해외주식 ETF로, 대형주 중심의 성장주에 투자합니다. 수수료는 0.05%이며, 최근 1년 동안 40

In [63]:
vectorstore = LangchainFAISS.from_documents(documents, embeddings)

In [64]:
vectorstore.index.ntotal

12

In [65]:
query = '미국 기술주에 투자하고 싶어요'
vectorstore.similarity_search_with_score(query, k=3)

[(Document(id='e8a7664a-bfaa-4305-94d5-08ca3fe364a4', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': np.float64(-10.35), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 AI 관련 기술주에 투자하는 고위험 상품으로, 최근 1년 동안 171.26%의 높은 수익률을 기록했습니다. 수수료는 0.49%이며, 배당수익률은 0%로 변동성이 31.35%로 나타나고 있습니다. 키워드: 반도체, AI, 미국, 기술주, 고위험수수료 0.49%, 배당수익률: 0.0%수익률 : 171.26% / 변동성: 31.35% / MDDD : -10.35%'),
  np.float32(1.1795318)),
 (Document(id='7dcceaac-ae1d-44cc-9148-447f5cb4e428', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(52.41), 'volatility': np.float64(16.57), 'mdd': np.float64(-7.34), 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국, 나스닥, 기술주, 성장

In [66]:
etf_knowledge_base = [
    {"ticker": "069500", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
     "risk": "중간", "expense_ratio": 0.15},
    {"ticker": "379800", "name": "KODEX 미국S&P500TR", "category": "해외주식",
     "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
     "risk": "중간", "expense_ratio": 0.05},
    {"ticker": "461460", "name": "KODEX 미국나스닥100TR", "category": "해외주식",
     "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
     "risk": "높음", "expense_ratio": 0.05},
]

In [67]:
etf_names = [e['name'] + ': ' + e['description'] for e in etf_knowledge_base]
etf_names

['KODEX 200: KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.',
 'KODEX 미국S&P500TR: S&P500 추적, 배당 자동 재투자. 수수료 0.05%.',
 'KODEX 미국나스닥100TR: 나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.']

In [68]:
prompt = f"""다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요

ETF 목록:
{json.dumps(etf_names, ensure_ascii=False, indent=2)}

형식 : 번호. 질문"""

synthetic = llm.invoke(prompt)

In [69]:
synthetic

AIMessage(content='1. KODEX 200 ETF는 어떤 종류의 주식에 투자하나요? 대형주 중심이라고 했는데, 구체적으로 어떤 기업들이 포함되어 있나요?\n\n2. KODEX 미국S&P500TR ETF의 배당금은 어떻게 처리되나요? 배당금이 자동으로 재투자된다고 했는데, 이로 인해 어떤 장점이 있나요?\n\n3. KODEX 미국나스닥100TR ETF는 높은 변동성을 가진다고 했는데, 이 ETF에 투자할 때 어떤 리스크를 고려해야 하나요?\n\n4. KODEX 200 ETF와 KODEX 미국S&P500TR ETF의 수수료 차이가 투자 성과에 어떤 영향을 미칠까요?\n\n5. KODEX 미국나스닥100TR ETF에 투자할 경우, 기술주 중심의 포트폴리오가 어떤 장점과 단점을 가질까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 191, 'prompt_tokens': 144, 'total_tokens': 335, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVGJpWmLAUrXfHKkRCWoDZRo6h32S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d964f-4100-78b3-

In [70]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

In [71]:
kiwi = Kiwi()

In [72]:
def kiwi_tokenize(text):
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        if token.tag in ("SL", "SN", "NNG", "NNP"):
            result.append(token.form.lower())
            
    return result

In [73]:
kiwi_tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?")

['kodex', '200', 'etf', '종류', '주식', '투자']

In [74]:
test_text = "배당수익률 3% 이상인 미국 ETF를 추천해주세요"
kiwi_tokenize(test_text)

['배당', '수익', '3', '이상', '미국', 'etf', '추천']

In [75]:
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]

In [76]:
bm25 = BM25Okapi(corpus)

In [78]:
def bm25_search(query, k=5):
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]
    

In [79]:
test_queries = [
    "KODEX 200",
    "안전한 투자 상품",
    "배당 ETF",
    "인플레이션 방어"
]

In [80]:
for q in test_queries:
    v_result = vectorstore.similarity_search(q, k=1)
    b_result = bm25_search(q, k=1)
    v_name = v_result[0].metadata['name'] if v_result else "-"
    b_name = b_result[0][0].metadata['name'] if b_result else "-"
    print(f"{q:20s} | {v_name:25s} | {b_name:25s}")

KODEX 200            | KODEX 200                 | KODEX 200                
안전한 투자 상품            | KODEX 단기채권PLUS            | KODEX 단기채권PLUS           
배당 ETF               | ACE KRX금현물                | ACE KRX금현물               
인플레이션 방어             | KIWOOM 국고채10년레버리지         | KODEX 골드선물(H)            


In [ ]:
# bm25 : TF-IDF

In [ ]:
# k1, b 

In [83]:
query = "배당 ETF"
tokens = kiwi_tokenize(query)
params = [(0.5, 0.25), (1.5, 0.75), (2.5, 0.9)]
for k1, b in params:
    bm25_temp = BM25Okapi(corpus, k1=k1, b=b)
    scores = bm25_temp.get_scores(tokens)
    top3 = np.argsort(scores)[::-1][:3]
    print(f"k1 = {k1}, b= {b}")
    for idx in top3:
        name = documents[idx].metadata['name']
        print(f" [{scores[idx]}] {name}")
    

k1 = 0.5, b= 0.25
 [0.8937365555378389] ACE KRX금현물
 [0.8919633776295501] TIGER 차이나전기차SOLACTIVE
 [0.8919633776295501] TIGER 2차전지테마
k1 = 1.5, b= 0.75
 [1.1595543275994418] ACE KRX금현물
 [1.052417980080111] TIGER 차이나전기차SOLACTIVE
 [1.052417980080111] TIGER 2차전지테마
k1 = 2.5, b= 0.9
 [1.3673477212129435] ACE KRX금현물
 [1.1393005502054363] TIGER 차이나전기차SOLACTIVE
 [1.1393005502054363] TIGER 2차전지테마


In [90]:
def filtered_search(vectorstore, query, filters=None, k=5, fetch_k=20):  # filters = {"expense_ratio" : 0.1, "category" : "해외주식"}
    results = vectorstore.similarity_search_with_score(query, fetch_k)
    if not filters:
        return results[:k]
    
    filtered = []
    for doc, score in results:
        match = True
        for key, condition in filters.items():
            val = doc.metadata.get(key)  # 0.49
            if isinstance(condition, dict):
                if "less_than" in condition and val > condition["less_than"]:
                    match = False
                if "greater_than" in condition and val < condition["greater_than"]:
                    match = False
            elif val != condition:
                match = False
        if match:
            filtered.append((doc, score))
    
    return filtered[:k]

In [91]:
results = filtered_search(vectorstore, "저비용 해외 ETF", 
                          filters = {"category":"해외주식", "expense_ratio" : {"less_than": 0.5}}, k=3)

In [92]:
results

[(Document(id='5f3d3742-a00a-4e8d-929a-fb49fc8cb52b', metadata={'ticker': '371460', 'name': 'TIGER 차이나전기차SOLACTIVE', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(71.71), 'volatility': np.float64(24.87), 'mdd': np.float64(-11.34), 'risk_level': '중간', 'data_staus': 'real', 'keywords': '중국, 전기차, 테마, 해외주식, 고위험'}, page_content='TIGER 차이나전기차SOLACTIVE (해외주식): TIGER 차이나전기차SOLACTIVE ETF는 중국 전기차 관련 주식에 투자하는 해외주식 ETF로, 고위험 고수익을 추구하며 1년 수익률이 71.71%에 달합니다. 수수료는 0.49%이며, 배당수익률은 0.0%입니다. 키워드: 중국, 전기차, 테마, 해외주식, 고위험수수료 0.49%, 배당수익률: 0.0%수익률 : 71.71% / 변동성: 24.87% / MDDD : -11.34%'),
  np.float32(1.0649977)),
 (Document(id='e8a7664a-bfaa-4305-94d5-08ca3fe364a4', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': np.float64(-10.35), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, p